[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-15-llm-app-operations-statistics.ipynb)

# Advanced Discussion 15 — latency percentiles, drift alarms, release gates and judge agreement

Simulated latency mixtures, ADWIN and PSI on a retrieval-similarity stream, a confidence-bound release gate, and Cohen's kappa.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds.

In [ ]:
%pip install -q river

In [ ]:
import numpy as np
from river import drift
rng = np.random.RandomState(0)

## 1. latency: why the mean lies. Simulated per-request latencies (seconds): fast cache hits + log-normal model calls + rare stalls

In [ ]:
def latencies(n, seed=0):
    r = np.random.RandomState(seed); hit = r.rand(n) < 0.25
    lat = np.where(hit, r.lognormal(np.log(0.05), 0.3, n), r.lognormal(np.log(1.6), 0.45, n))
    stall = r.rand(n) < 0.01; lat[stall] += r.uniform(8, 20, stall.sum()); return lat
lat = latencies(50000)
print("latency over 50,000 requests: mean %.2fs | p50 %.2fs | p95 %.2fs | p99 %.2fs | max %.1fs" % (lat.mean(), *np.percentile(lat, [50, 95, 99]), lat.max()))
print("share of requests slower than 3 s: %.1f%%" % (100 * (lat > 3).mean()))
print("a 4-step agent-style chain (sum of 4 calls): p50 %.1fs, p95 %.1fs" % tuple(np.percentile(sum(latencies(50000, s) for s in range(4)), [50, 95])))

## 2. drift in a retrieval-quality signal: top-1 similarity of each query to the corpus

In [ ]:
stream = np.r_[rng.normal(0.62, 0.08, 1500), rng.normal(0.55, 0.09, 1500)]            # the corpus goes stale at request 1500 (new product names)
adwin = drift.ADWIN(delta=0.002); first = None
for t, x in enumerate(stream):
    adwin.update(x)
    if adwin.drift_detected and t >= 1500 and first is None: first = t
print("\nADWIN on top-1 similarity: drift flagged at request %s (true change at 1500)" % first)
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected); a = np.histogram(actual, edges)[0] / len(actual); e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(((a - e) * np.log(a / e)).sum())
print("PSI of the last 500 requests vs the reference window: before the change %.3f | after the change %.3f  (common reading: <0.1 stable, 0.1-0.25 moderate, >0.25 large)" % (psi(stream[:1000], stream[1000:1500]), psi(stream[:1000], stream[2500:3000])))

## 3. a release gate for a prompt/model change: candidate must not be worse, with confidence

In [ ]:
def gate(base, cand, margin=0.02, iters=4000, seed=0):
    r = np.random.RandomState(seed); d = cand.astype(float) - base.astype(float); n = len(d)
    lo = np.percentile([d[r.randint(0, n, n)].mean() for _ in range(iters)], 2.5); return d.mean(), lo, lo > -margin
for n, pa, pb in ((200, 0.80, 0.81), (200, 0.80, 0.74), (1000, 0.80, 0.81), (1000, 0.80, 0.77)):
    r = np.random.RandomState(n + int(pb * 100)); u = r.rand(n); base = (u < pa).astype(int); cand = (np.clip(u + r.normal(0, 0.08, n), 0, 1) < pb).astype(int)
    m, lo, ok = gate(base, cand); print("  n=%4d true %.2f -> %.2f: observed change %+.3f, lower 95%% bound %+.3f -> %s" % (n, pa, pb, m, lo, "PASS" if ok else "BLOCK"))

## 4. judge agreement: Cohen's kappa between an automatic judge and human labels

In [ ]:
def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); po = (a == b).mean(); pe = sum((a == c).mean() * (b == c).mean() for c in (0, 1)); return (po - pe) / (1 - pe)
human = np.r_[np.ones(70), np.zeros(30)].astype(int)                        # 70 good answers, 30 bad
judge = human.copy(); flip_good = rng.choice(70, 6, replace=False); judge[flip_good] = 0; flip_bad = 70 + rng.choice(30, 12, replace=False); judge[flip_bad] = 1
print("\njudge vs human on 100 answers: raw agreement %.2f, kappa %.2f  (judge passes %.0f%% vs humans %.0f%%)" % ((judge == human).mean(), kappa(judge, human), 100 * judge.mean(), 100 * human.mean()))
print("a judge that always says 'good' agrees %.2f of the time but kappa is %.2f" % (((np.ones(100, int)) == human).mean(), kappa(np.ones(100, int), human)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.